# Held-out SciFact Cascade Evaluation (Kaggle)

This notebook runs the corrected evaluation. It uses SciFact train qrels for threshold selection and test qrels for final evaluation. The test path executes BM25 first and conditionally skips Dense/SPLADE on early exit.

Before running, upload `cascade_source_bundle.zip` from this project as a Kaggle dataset and attach it as notebook input. Enable Internet so BEIR data and the model weights can be downloaded, and select a GPU accelerator (a single T4 is sufficient for this SciFact run).

In [ ]:
!pip install -q beir ranx rank-bm25 faiss-cpu sentence-transformers transformers scipy tqdm numpy


In [ ]:
import os
from pathlib import Path
from shutil import copytree
from zipfile import ZipFile
input_dir = Path("/kaggle/input")
working_dir = Path("/kaggle/working")
project_name = "cascaded-blended-rag"
project_dir = working_dir / project_name
def find_named_files(root, filename):
    matches = []
    for current, dirs, files in os.walk(root, followlinks=True):
        if filename in files:
            matches.append(Path(current) / filename)
    return matches
bundles = find_named_files(input_dir, "cascade_source_bundle.zip")
if bundles:
    source_bundle = sorted(bundles)[0]
    with ZipFile(source_bundle) as archive:
        archive.extractall(working_dir)
    print("Extracted source bundle:", source_bundle)
else:
    scripts = find_named_files(input_dir, "08_evaluate_cascade_heldout.py")
    if len(scripts) == 1:
        source_project = scripts[0].parents[1]
        copytree(source_project, project_dir, dirs_exist_ok=True)
        print("Copied extracted source tree:", source_project)
    else:
        visible = [str(p) for p in input_dir.iterdir()] if input_dir.exists() else []
        raise FileNotFoundError(
            f"Could not find exactly one source zip or evaluation script. "
            f"Attached top-level inputs: {visible}. Attach the cascade-source-bundle dataset using Add Input."
        )
assert (project_dir / "experiments/08_evaluate_cascade_heldout.py").exists(), project_dir
print("Project directory:", project_dir)


In [ ]:
%cd /kaggle/working/cascaded-blended-rag
!python experiments/08_evaluate_cascade_heldout.py


In [ ]:
from IPython.display import FileLink, display
result_path = Path("/kaggle/working/cascaded-blended-rag/results/cascade_heldout_scifact.json")
if result_path.exists():
    display(FileLink(str(result_path)))
else:
    print("Result JSON was not produced; inspect the experiment cell output.")
